In [29]:
import torch.nn as nn
import pandas as pd
import json
import numpy as np

In [30]:
class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_dim: int = 1,
        hidden_dim: int = 64,
        latent_dim: int = 32,
        num_layers: int = 1,
    ):
        
        super().__init__()
        
        # ---- Encoder ---- #
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        
        # ---- Latent ---- #
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)
        
        # ---- Decoder ---- #
        self.decoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        
    # ---- Forward ---- #
    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        hidden = hidden[-1]
        latent = self.to_latent(hidden)
        hidden_dec = self.from_latent(latent)
        seq_len = x.size(1)
        repeated_hidden = hidden_dec.unsqueeze(1).repeat(1, seq_len, 1)
        reconstructed, _ = self.decoder(repeated_hidden)
        return reconstructed

In [32]:
def load_series(csv_path: str):
    df = pd.read_csv(csv_path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df

load_series('../assets/data/art_daily_flatmiddle.csv')

,timestamp,value
0,2014-04-01 00:00:00,-21.048383
1,2014-04-01 00:05:00,-20.295477
2,2014-04-01 00:10:00,-18.127229
3,2014-04-01 00:15:00,-20.171665
4,2014-04-01 00:20:00,-21.223762
...,...,...
4027,2014-04-14 23:35:00,-18.083562
4028,2014-04-14 23:40:00,-20.278406
4029,2014-04-14 23:45:00,-20.063239
4030,2014-04-14 23:50:00,-20.751973


In [ ]:
def load_windows(json_path: str, file_key: str):
    with open(json_path, "r") as f:
        data = json.load(f)
    return data[file_key]

In [ ]:


def build_labels(df, windows):
    labels = np.zeros(len(df))
    for start, end in windows:
        start = pd.to_datetime(start)
        end = pd.to_datetime(end)
        
        mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)
        labels[mask] = 1

    df["labels"] = labels
    return df

In [28]:
def make_windows(values, seq_len):
    windows = []

    for i in range(len(values) - seq_len):
        window = values[i : i + seq_len]
        windows.append(window)

    return np.array(windows)